In [7]:
import pandas as pd
import numpy as np

WINDOW = 14          # days per window (matches spec: fenêtres temporelles glissantes)
HORIZON = 7           # predict failure within this many days after the window ends
STRIDE_FAILED = 1     # dense sampling for failed drives — we need all the signal we can get
STRIDE_HEALTHY = 7    # sparser sampling for healthy drives — controls class imbalance

data = pd.read_csv("../data/processed/st12000nm0008_2026q1q2_clean.csv.gz",
                   parse_dates=["date"])
fail_dates = pd.read_csv("../data/processed/fail_dates.csv", parse_dates=["fail_date"])
split_map = pd.read_csv("../data/processed/drive_split.csv")

fail_date_lookup = dict(zip(fail_dates["serial_number"], fail_dates["fail_date"]))
split_lookup = dict(zip(split_map["serial_number"], split_map["split"]))

FEATURE_COLS = [c for c in data.columns if c.startswith("smart_")]
print("features used:", len(FEATURE_COLS))
print(FEATURE_COLS)

KeyboardInterrupt: 

In [2]:
def make_windows_for_drive(df_drive, fail_date, window=WINDOW, horizon=HORIZON, stride=1):
    """
    df_drive: rows for ONE drive, already sorted by date.
    fail_date: the drive's failure date, or None if it never failed.
    Returns lists of (feature_array, label, window_end_date).
    """
    df_drive = df_drive.sort_values("date").reset_index(drop=True)
    n = len(df_drive)
    windows, labels, end_dates = [], [], []

    for start in range(0, n - window + 1, stride):
        end = start + window - 1
        window_dates = df_drive["date"].iloc[start:end + 1]

        # Skip windows with calendar gaps (missing days) — a 14-row window
        # spanning more than 13 calendar days is not a clean 14-day window.
        if (window_dates.iloc[-1] - window_dates.iloc[0]).days != window - 1:
            continue

        window_end_date = window_dates.iloc[-1]
        label = 0
        if fail_date is not None:
            delta = (fail_date - window_end_date).days
            if 1 <= delta <= horizon:
                label = 1

        feat = df_drive.loc[start:end, FEATURE_COLS].values.astype(np.float32)
        windows.append(feat)
        labels.append(label)
        end_dates.append(window_end_date)

    return windows, labels, end_dates

In [3]:
results = {"train": [], "val": [], "test": []}

for serial, df_drive in data.groupby("serial_number"):
    split = split_lookup.get(serial)
    if split is None:
        continue  # shouldn't happen, but guards against mismatched data

    fail_date = fail_date_lookup.get(serial)  # None if healthy
    stride = STRIDE_FAILED if fail_date is not None else STRIDE_HEALTHY

    windows, labels, end_dates = make_windows_for_drive(df_drive, fail_date, stride=stride)
    for w, l in zip(windows, labels):
        results[split].append((w, l))

for split in ["train", "val", "test"]:
    n_total = len(results[split])
    n_pos = sum(l for _, l in results[split])
    print(f"{split}: {n_total} windows | positive (label=1): {n_pos} "
          f"| ratio 1:{(n_total - n_pos) / max(n_pos, 1):.0f}")

train: 314332 windows | positive (label=1): 713 | ratio 1:440
val: 67570 windows | positive (label=1): 173 | ratio 1:390
test: 67675 windows | positive (label=1): 158 | ratio 1:427


In [4]:
# Subsample healthy (label=0) windows down to a target ratio per split,
# keeping ALL positive (label=1) windows intact.
TARGET_RATIO = 30   # negatives per positive — adjust here if needed, nowhere else

rng = np.random.default_rng(42)

def subsample_split(split_windows, target_ratio, rng):
    positives = [(w, l) for w, l in split_windows if l == 1]
    negatives = [(w, l) for w, l in split_windows if l == 0]

    n_target_neg = min(len(negatives), len(positives) * target_ratio)
    idx = rng.choice(len(negatives), size=n_target_neg, replace=False)
    sampled_negatives = [negatives[i] for i in idx]

    combined = positives + sampled_negatives
    rng.shuffle(combined)
    return combined

for split in ["train", "val", "test"]:
    before = len(results[split])
    results[split] = subsample_split(results[split], TARGET_RATIO, rng)
    after = len(results[split])
    n_pos = sum(l for _, l in results[split])
    print(f"{split}: {before} -> {after} windows | positive: {n_pos} "
          f"| ratio 1:{(after - n_pos) / max(n_pos, 1):.0f}")

train: 314332 -> 22103 windows | positive: 713 | ratio 1:30
val: 67570 -> 5363 windows | positive: 173 | ratio 1:30
test: 67675 -> 4898 windows | positive: 158 | ratio 1:30


In [5]:
def stack_split(split_name):
    X = np.stack([w for w, _ in results[split_name]])   # (n_windows, 14, n_features)
    y = np.array([l for _, l in results[split_name]], dtype=np.float32)
    return X, y

X_train, y_train = stack_split("train")
X_val, y_val     = stack_split("val")
X_test, y_test   = stack_split("test")

# Fit normalization on TRAIN ONLY — this is the leakage guard for Obj. 2.
feat_mean = X_train.reshape(-1, X_train.shape[-1]).mean(axis=0)
feat_std  = X_train.reshape(-1, X_train.shape[-1]).std(axis=0)
feat_std[feat_std == 0] = 1.0  # avoid divide-by-zero on constant columns

def normalize(X):
    return (X - feat_mean) / feat_std

X_train_n = normalize(X_train)
X_val_n   = normalize(X_val)
X_test_n  = normalize(X_test)

print("train shape:", X_train_n.shape, "| val:", X_val_n.shape, "| test:", X_test_n.shape)

train shape: (22103, 14, 22) | val: (5363, 14, 22) | test: (4898, 14, 22)


In [6]:
np.savez_compressed("../data/processed/windows_train.npz", X=X_train_n, y=y_train)
np.savez_compressed("../data/processed/windows_val.npz",   X=X_val_n,   y=y_val)
np.savez_compressed("../data/processed/windows_test.npz",  X=X_test_n,  y=y_test)

# Save the normalization stats too — the Django backend will need these
# at inference time to normalize live data the same way.
np.savez("../data/processed/normalization_stats.npz",
          mean=feat_mean, std=feat_std, feature_cols=FEATURE_COLS)

print("saved.")

saved.
